# v63 — Auger UHECR Quadrupole Test

**Standard KTF³ (v52):** Dipole 22.9° from KTF³ axis, σ=-0.72 NULL

**NEW Pin⁻ prediction:**
A Twisted Klein Bottle has BOTH:
- Dipole (from Z₂ reflection) — already tested
- Quadrupole (from chiral twist θ_R) — NEW test

The quadrupole should be oriented WITH the KTF³ axis
and rotated by θ_R = 25.7° relative to the dipole.

**KTF³ axis:** l=277°, b=-31°
**Dipole axis (Auger):** l≈233°, b≈-13° (observed)
**Predicted quadrupole axis:** rotated by 25.7° from dipole

**Data:** Pierre Auger Observatory public UHECR catalog

In [ ]:
!pip install numpy scipy matplotlib astropy healpy -q
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize
from astropy.coordinates import SkyCoord, Galactic
import astropy.units as u
import urllib.request, os
print('Ready.')

In [ ]:
# KTF³ parameters
l_KTF3  = 277.0   # degrees galactic
b_KTF3  = -31.0   # degrees galactic
theta_R = 25.7    # degrees — Pin⁻ twist angle

# Auger observed dipole (from Auger 2017 paper)
l_dipole = 233.0  # degrees galactic
b_dipole = -13.0  # degrees galactic
d_amplitude = 0.065  # dipole amplitude

# Pin⁻ prediction: quadrupole axis rotated by theta_R from dipole
# The rotation is around the KTF³ axis
l_quad_pred = l_dipole + theta_R  # simplified prediction
b_quad_pred = b_dipole

# Convert all to equatorial
def gal_to_eq(l, b):
    c = SkyCoord(l=l*u.deg, b=b*u.deg, frame='galactic')
    return c.icrs.ra.deg, c.icrs.dec.deg

ra_KTF3,  dec_KTF3  = gal_to_eq(l_KTF3, b_KTF3)
ra_dip,   dec_dip   = gal_to_eq(l_dipole, b_dipole)
ra_quad,  dec_quad  = gal_to_eq(l_quad_pred, b_quad_pred)

print('=== AXES ===')
print(f'KTF³ axis:        l={l_KTF3}°, b={b_KTF3}°')
print(f'Auger dipole:     l={l_dipole}°, b={b_dipole}°')
print(f'Predicted quad:   l={l_quad_pred}°, b={b_quad_pred}°')
print()
print(f'Separation KTF³-Dipole: {abs(l_KTF3-l_dipole):.1f}° in l')
print(f'Pin⁻ predicts quadrupole at {theta_R}° from dipole')

In [ ]:
# Download Auger public data
# Public catalog from Auger Observatory
AUGER_URL = 'https://www.auger.org/document-centre/analysis-data/public-data/'

# Try multiple sources
urls = [
    'https://www.auger.org/index.php/document-centre/finish/107-data/4822-auger-2021-data-release',
    'https://raw.githubusercontent.com/AUGER-public-data/UHECR/main/events.csv',
]

AUGER_FILE = 'auger_uhecr.csv'
downloaded = False

if not os.path.exists(AUGER_FILE):
    for url in urls:
        try:
            urllib.request.urlretrieve(url, AUGER_FILE)
            print(f'Downloaded from: {url}')
            downloaded = True
            break
        except Exception as e:
            print(f'Failed: {e}')

# If no download, use synthetic data based on Auger statistics
if not os.path.exists(AUGER_FILE) or os.path.getsize(AUGER_FILE) < 100:
    print('Using synthetic Auger-like data (Auger 2022: ~2500 events >8 EeV)')
    USE_SYNTHETIC = True
    np.random.seed(42)
    N_events = 2500

    # Generate isotropic base + observed dipole
    # Auger dipole: d=0.065 at l=233°, b=-13°
    ra_dip_rad  = np.radians(ra_dip)
    dec_dip_rad = np.radians(dec_dip)

    # Sample from dipole distribution
    ra_ev  = np.random.uniform(0, 360, N_events)
    dec_ev = np.degrees(np.arcsin(np.random.uniform(-1, 0, N_events)))  # Auger sees southern sky

    # Add dipole modulation
    ra_rad  = np.radians(ra_ev)
    dec_rad = np.radians(dec_ev)
    cos_angle = (np.sin(dec_rad)*np.sin(dec_dip_rad) +
                 np.cos(dec_rad)*np.cos(dec_dip_rad)*np.cos(ra_rad-ra_dip_rad))
    prob = 1 + d_amplitude * cos_angle
    prob = prob / prob.max()
    keep = np.random.uniform(0, 1, N_events) < prob
    ra_ev  = ra_ev[keep]
    dec_ev = dec_ev[keep]

    energy_ev = np.random.uniform(8, 100, len(ra_ev))  # EeV
    print(f'Synthetic events: {len(ra_ev)}')
else:
    USE_SYNTHETIC = False
    print('Loading real Auger data...')

In [ ]:
# === MULTIPOLE DECOMPOSITION ===
# Decompose UHECR arrival directions into dipole + quadrupole

def angular_sep(ra1, dec1, ra2, dec2):
    """Angular separation in degrees."""
    r1, d1 = np.radians(ra1), np.radians(dec1)
    r2, d2 = np.radians(ra2), np.radians(dec2)
    cos_sep = np.sin(d1)*np.sin(d2) + np.cos(d1)*np.cos(d2)*np.cos(r1-r2)
    return np.degrees(np.arccos(np.clip(cos_sep, -1, 1)))

# Convert to cartesian
ra_r  = np.radians(ra_ev)
dec_r = np.radians(dec_ev)
x = np.cos(dec_r) * np.cos(ra_r)
y = np.cos(dec_r) * np.sin(ra_r)
z = np.sin(dec_r)

# Dipole: <x>, <y>, <z>
dx, dy, dz = x.mean(), y.mean(), z.mean()
d_amp = np.sqrt(dx**2 + dy**2 + dz**2)
d_ra  = np.degrees(np.arctan2(dy, dx)) % 360
d_dec = np.degrees(np.arcsin(dz/d_amp))

# Convert to galactic
d_coord = SkyCoord(ra=d_ra*u.deg, dec=d_dec*u.deg, frame='icrs')
d_l = d_coord.galactic.l.deg
d_b = d_coord.galactic.b.deg

print('=== DIPOLE ===')
print(f'Measured dipole: l={d_l:.1f}°, b={d_b:.1f}°, amplitude={d_amp:.4f}')
print(f'Auger observed:  l={l_dipole}°, b={b_dipole}°, amplitude={d_amplitude}')
print()

# Quadrupole: <x²-1/3>, <y²-1/3>, <z²-1/3>, <xy>, <xz>, <yz>
Qxx = (x**2 - 1/3).mean()
Qyy = (y**2 - 1/3).mean()
Qzz = (z**2 - 1/3).mean()
Qxy = (x*y).mean()
Qxz = (x*z).mean()
Qyz = (y*z).mean()

Q = np.array([[Qxx, Qxy, Qxz],
               [Qxy, Qyy, Qyz],
               [Qxz, Qyz, Qzz]])

# Eigenvalues and eigenvectors of quadrupole tensor
eigenvalues, eigenvectors = np.linalg.eigh(Q)
Q_amp = eigenvalues.max() - eigenvalues.min()

# Principal axis (largest eigenvalue)
principal = eigenvectors[:, eigenvalues.argmax()]
q_ra  = np.degrees(np.arctan2(principal[1], principal[0])) % 360
q_dec = np.degrees(np.arcsin(principal[2]))

q_coord = SkyCoord(ra=q_ra*u.deg, dec=q_dec*u.deg, frame='icrs')
q_l = q_coord.galactic.l.deg
q_b = q_coord.galactic.b.deg

print('=== QUADRUPOLE ===')
print(f'Measured quadrupole axis: l={q_l:.1f}°, b={q_b:.1f}°')
print(f'Predicted (Pin⁻):         l={l_quad_pred}°, b={b_quad_pred}°')
print(f'Quadrupole amplitude: {Q_amp:.4f}')
print()

# Angular separation between measured and predicted quadrupole
sep = angular_sep(q_ra, q_dec, ra_quad, dec_quad)
sep2 = angular_sep(q_ra, q_dec, 360-ra_quad, -dec_quad)  # antipodal
sep_min = min(sep, sep2)
print(f'Separation from Pin⁻ prediction: {sep_min:.1f}°')
print(f'(Within 30° = consistent with Pin⁻)')

In [ ]:
# === MONTE CARLO: Is quadrupole signal significant? ===
N_MC = 1000
mc_Q_amps = []
mc_seps = []

print(f'Running {N_MC} MC simulations...')
for i in range(N_MC):
    # Isotropic null + Auger dipole only
    np.random.seed(i)
    n = len(ra_ev)
    ra_mc  = np.random.uniform(0, 360, n)
    dec_mc = np.degrees(np.arcsin(np.random.uniform(-1, 0, n)))

    # Add only dipole
    ra_r_mc  = np.radians(ra_mc)
    dec_r_mc = np.radians(dec_mc)
    ra_d_r   = np.radians(ra_dip)
    dec_d_r  = np.radians(dec_dip)
    cos_a = (np.sin(dec_r_mc)*np.sin(dec_d_r) +
             np.cos(dec_r_mc)*np.cos(dec_d_r)*np.cos(ra_r_mc-ra_d_r))
    prob = 1 + d_amplitude * cos_a
    prob = prob / prob.max()
    keep = np.random.uniform(0,1,n) < prob
    x_mc = np.cos(dec_r_mc[keep])*np.cos(ra_r_mc[keep])
    y_mc = np.cos(dec_r_mc[keep])*np.sin(ra_r_mc[keep])
    z_mc = np.sin(dec_r_mc[keep])

    if len(x_mc) < 10:
        continue

    Qxx_mc = (x_mc**2-1/3).mean()
    Qyy_mc = (y_mc**2-1/3).mean()
    Qzz_mc = (z_mc**2-1/3).mean()
    Qxy_mc = (x_mc*y_mc).mean()
    Qxz_mc = (x_mc*z_mc).mean()
    Qyz_mc = (y_mc*z_mc).mean()
    Q_mc = np.array([[Qxx_mc,Qxy_mc,Qxz_mc],[Qxy_mc,Qyy_mc,Qyz_mc],[Qxz_mc,Qyz_mc,Qzz_mc]])
    ev_mc, evec_mc = np.linalg.eigh(Q_mc)
    mc_Q_amps.append(ev_mc.max()-ev_mc.min())

mc_Q_amps = np.array(mc_Q_amps)
sigma_quad = (Q_amp - mc_Q_amps.mean()) / mc_Q_amps.std()

print(f'\nQuadrupole amplitude: {Q_amp:.4f}')
print(f'MC mean: {mc_Q_amps.mean():.4f} ± {mc_Q_amps.std():.4f}')
print(f'Significance: σ = {sigma_quad:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('v63 — Auger UHECR Quadrupole Test (Pin⁻ KTF³)', fontweight='bold')

ax = axes[0]
ax.hist(mc_Q_amps, bins=30, alpha=0.7, color='gray', label='MC null (dipole only)')
ax.axvline(Q_amp, color='red', lw=2.5, label=f'Measured Q={Q_amp:.4f}')
ax.axvline(mc_Q_amps.mean(), color='blue', ls='--', label='MC mean')
ax.set_xlabel('Quadrupole amplitude')
ax.set_ylabel('Count')
ax.set_title(f'Quadrupole significance: σ={sigma_quad:.2f}')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
# Sky map in galactic coordinates
# Plot event density and key axes
l_ev_arr = []
for r, d in zip(ra_ev, dec_ev):
    c = SkyCoord(ra=r*u.deg, dec=d*u.deg, frame='icrs')
    l_ev_arr.append(c.galactic.l.deg)
l_ev_arr = np.array(l_ev_arr)

ax.scatter(l_ev_arr, dec_ev, alpha=0.1, s=1, color='gray', label='Events')
ax.scatter(l_KTF3, b_KTF3, s=200, c='blue', marker='*', zorder=5, label=f'KTF³ axis ({l_KTF3}°,{b_KTF3}°)')
ax.scatter(l_dipole, b_dipole, s=200, c='red', marker='^', zorder=5, label=f'Dipole ({l_dipole}°,{b_dipole}°)')
ax.scatter(l_quad_pred, b_quad_pred, s=200, c='green', marker='D', zorder=5, label=f'Predicted Quad ({l_quad_pred}°)')
ax.scatter(q_l, q_b, s=200, c='orange', marker='s', zorder=5, label=f'Measured Quad ({q_l:.0f}°,{q_b:.0f}°)')
ax.set_xlabel('Galactic longitude l [°]')
ax.set_ylabel('Galactic latitude b [°]')
ax.set_title('Sky map: Axes comparison')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('v63_auger_quadrupole.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v63_auger_quadrupole.png')

In [ ]:
print('=' * 60)
print('v63 SUMMARY — Auger Quadrupole Test')
print('=' * 60)
print()
print('Pin⁻ Prediction:')
print(f'  Quadrupole axis at l={l_quad_pred}°, b={b_quad_pred}°')
print(f'  (Dipole l={l_dipole}° + θ_R={theta_R}° twist)')
print()
print('Measurement:')
print(f'  Quadrupole axis: l={q_l:.1f}°, b={q_b:.1f}°')
print(f'  Amplitude: {Q_amp:.4f}')
print(f'  Significance: σ={sigma_quad:.2f}')
print(f'  Separation from prediction: {sep_min:.1f}°')
print()
if USE_SYNTHETIC:
    print('⚠️  SYNTHETIC DATA — pipeline test only!')
    print('  Run with real Auger data for actual result.')
    print('  Real Auger data: https://www.auger.org/data')
else:
    if sigma_quad > 2 and sep_min < 30:
        print('✅ SIGNAL: Quadrupole significant AND aligned with prediction!')
    elif sigma_quad > 2:
        print('⚠️ PARTIAL: Quadrupole significant but misaligned')
    else:
        print('❌ NULL: Quadrupole not significant')
print()
print('Standard KTF³ (v52): σ=-0.72 NULL for dipole')
print('Pin⁻ TKB: quadrupole is the NEW prediction')
print()
print('OSF: osf.io/42nks | GitHub: github.com/Andrew-Cot/KTF3-Analyse')